## Overview

[Gemini](https://ai.google.dev/models/gemini) is a family of generative AI models that lets developers generate content and solve problems. These models are designed and trained to handle both text and images as input.

[LangChain](https://www.langchain.com/) is a data framework designed to make integration of Large Language Models (LLM) like Gemini easier for applications.

[Chroma](https://docs.trychroma.com/) is an open-source embedding database focused on simplicity and developer productivity. Chroma allows users to store embeddings and their metadata, embed documents and queries, and search the embeddings quickly.

In this notebook, you'll learn how to create an application that answers questions using data from a website with the help of Gemini, LangChain, and Chroma.

## Setup

First, you must install the packages and set the necessary environment variables.

### Installation

Install LangChain's Python library, `langchain` and LangChain's integration package for Gemini, `langchain-google-genai`. Next, install Chroma's Python client SDK, `chromadb`.

In [1]:
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough
from bs4 import BeautifulSoup

c:\Users\utcpret\Desktop\UTC\AI31\Projet RAG\AI31_Projet_RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\utcpret\AppData\Local\Temp\ipykernel_65136\1255822101.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


## Configure your API key

To run the following cell, your API key must be stored in a Colab Secret named `GOOGLE_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb) for an example.


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

### Read and parse the website data

LangChain provides a wide variety of document loaders. To read the website data as a document, you will use the `WebBaseLoader` from LangChain.

To know more about how to read and parse input data from different sources using the document loaders of LangChain, read LangChain's [document loaders guide](https://python.langchain.com/docs/integrations/document_loaders).

In [3]:

urls = [
    # # Texte intégral officiel — les 99 articles
     "https://eur-lex.europa.eu/legal-content/FR/TXT/HTML/?uri=CELEX:32016R0679",
 
]

loader = WebBaseLoader(urls)
docs = loader.load()
print(f"{len(docs)} documents chargés")

1 documents chargés


In [4]:
from langchain_community.document_loaders import PyPDFLoader

# recupération du pdf 
loader = PyPDFLoader("rgpd.pdf")
documents = loader.load()

# chunking a 1000 caractères 
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150) # chunk_overlap=150 => les 150 derniers caractères du chunk n sont répétés au début du chunk n+1 pour ne pas perdre de l'information
chunks = splitter.split_documents(documents) # 526 

# Nettoyage
chunks = [c for c in chunks if c.page_content.strip() != ""]

# test
print(f"Nombre de chunks : {len(chunks)}")
print(f"Taille chunk 0 : {len(chunks[0].page_content)} caractères")
print(chunks[0].page_content)

Nombre de chunks : 526
Taille chunk 0 : 959 caractères
4.5.2016   FR Journal officiel de l'Union européenne L 119/1
RÈGLEMENT (UE) 2016/679 DU PARLEMENT EUROPÉEN ET DU CONSEIL
du 27 avril 2016
relatif à la protection des personnes physiques à l'égard du traitement des données à caractère
personnel et à la libre circulation de ces données, et abrogeant la directive 95/46/CE (règlement
général sur la protection des données)
(Texte présentant de l'intérêt pour l'EEE)
LE PARLEMENT EUROPÉEN ET LE CONSEIL DE L'UNION EUROPÉENNE,
vu le traité sur le fonctionnement de l'Union européenne, et notamment son article 16,
vu la proposition de la Commission européenne,
après transmission du projet d'acte législatif aux parlements nationaux,
vu l'avis du Comité économique et social européen (1),
vu l'avis du Comité des régions (2),
statuant conformément à la procédure législative ordinaire (3),
considérant ce qui suit:
(1)La protection des personnes physiques à l'égard du traitement des données à carac

### Initialize Gemini's embedding model

To create the embeddings from the website data, you'll use Gemini's embedding model, **gemini-embedding-001** which supports creating text embeddings.

To use this embedding model, you have to import `GoogleGenerativeAIEmbeddings` from LangChain. To know more about the embedding model, read Google AI's [language documentation](https://ai.google.dev/models/gemini).

In [5]:
# afficher les models disponibles 
import google.generativeai as genai


genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

for model in genai.list_models():
    if "embedContent" in model.supported_generation_methods:
        print(model.name)

C:\Users\utcpret\AppData\Local\Temp\ipykernel_65136\1275663580.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)

c:\Users\utcpret\Desktop\UTC\AI31\Projet RAG\AI31_Projet_RAG\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\utcpret\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199

### Store the data using Chroma

To create a Chroma vector database from the website data, you will use the `from_documents` function of `Chroma`. Under the hood, this function creates embeddings from the documents created by the document loader of LangChain using any specified embedding model and stores them in a Chroma vector database.  

You have to specify the `docs` you created from the website data using LangChain's `WebBasedLoader` and the `gemini_embeddings` as the embedding model when invoking the `from_documents` function to create the vector database from the website data. You can also specify a directory in the `persist_directory` argument to store the vector store on the disk. If you don't specify a directory, the data will be ephemeral in-memory.


In [7]:
import time

# On indexe par petits lots avec une pause entre chaque
BATCH_SIZE = 50  # 50 chunks à la fois
all_chunks = chunks  #  chunks du chunking

vectordb = None
for i in range(0, len(all_chunks), BATCH_SIZE):
    batch = all_chunks[i:i + BATCH_SIZE]
    print(f"Indexation chunks {i} → {i + len(batch)}...")
    try :  
        if vectordb is None:
            vectordb = Chroma.from_documents(
                documents=batch,
                embedding=embeddings,
                persist_directory="./chroma_rgpd"
            )
        else:
            vectordb.add_documents(batch)

        print(f"OK — {vectordb._collection.count()} chunks indexés au total")
        time.sleep(20)  # pause 12 secondes entre chaque lot
        
    except Exception as e:
        print(f"Erreur au batch {i} : {e}")
        print("Attente 60 secondes avant de réessayer...")
        time.sleep(60)

print(f"Terminé — {vectordb._collection.count()} chunks indexés")

Indexation chunks 0 → 50...
OK — 50 chunks indexés au total
Indexation chunks 50 → 100...
OK — 100 chunks indexés au total
Indexation chunks 100 → 150...
OK — 150 chunks indexés au total
Indexation chunks 150 → 200...
OK — 200 chunks indexés au total
Indexation chunks 200 → 250...
OK — 250 chunks indexés au total
Indexation chunks 250 → 300...
OK — 300 chunks indexés au total
Indexation chunks 300 → 350...
OK — 350 chunks indexés au total
Indexation chunks 350 → 400...
OK — 400 chunks indexés au total
Indexation chunks 400 → 450...
OK — 450 chunks indexés au total
Indexation chunks 450 → 500...
OK — 500 chunks indexés au total
Indexation chunks 500 → 526...
OK — 526 chunks indexés au total
Terminé — 526 chunks indexés


### Create a retriever using Chroma

Le retriever transforme la question en vecteur, puis compare ce vecteur aux vecteurs de ChromaDB pour retrouver les chunks les plus proches

To load the vector store that you previously stored in the disk, you can specify the name of the directory that contains the vector store in `persist_directory` and the embedding model in the `embedding_function` arguments of Chroma's initializer.

You can then invoke the `as_retriever` function of `Chroma` on the vector store to create a retriever.

In [9]:
# Load from disk
vectorstore_disk = Chroma(
                        persist_directory="./chroma_db",       # Directory of db
                        embedding_function=embeddings   # Embedding model
                   )
# retriever 
retriever = vectorstore_disk.as_retriever(search_kwargs={"k": 4}, search_type = "mmr")

# Pour le RGPD, "mmr" est meilleur — un article peut apparaître dans plusieurs chunks proches, MMR évite la redondance
# le nombre de chunks à retourner. 4 est une bonne valeur pour le RGPD — assez pour couvrir une question qui touche plusieurs articles, pas trop pour ne pas noyer le LLM


## Generator

The Generator prompts the LLM for an answer when the user asks a question. The retriever you created in the previous stage from the Chroma vector store will be used to pass relevant embeddings from the website data to the LLM to provide more context to the user's query.

You'll perform the following steps in this stage:

1. Chain together the following:
    * A prompt for extracting the relevant embeddings using the retriever.
    * A prompt for answering any question using LangChain.
    * An LLM model from Gemini for prompting.
    
2. Run the created chain with a question as input to prompt the model for an answer.


### Initialize Gemini

You must import `ChatGoogleGenerativeAI` from LangChain to initialize your model.
 In this example, you will use **gemini-2.0-flash**, as it supports text summarization. To know more about the text model, read Google AI's [language documentation](https://ai.google.dev/models/gemini).

You can configure the model parameters such as ***temperature*** or ***top_p***,  by passing the appropriate values when initializing the `ChatGoogleGenerativeAI` LLM.  To learn more about the parameters and their uses, read Google AI's [concepts guide](https://ai.google.dev/docs/concepts#model_parameters).

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# To configure model parameters use the `generation_config` parameter.
# eg. generation_config = {"temperature": 0.7, "topP": 0.8, "topK": 40}
# If you only want to set a custom temperature for the model use the
# "temperature" parameter directly.

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")

### Create prompt templates

You'll use LangChain's [PromptTemplate](https://python.langchain.com/docs/how_to/#prompt-templates) to generate prompts to the LLM for answering questions.

In the `llm_prompt`, the variable `question` will be replaced later by the input question, and the variable `context` will be replaced by the relevant text from the website retrieved from the Chroma vector store.

In [ ]:
# Prompt template to query Gemini
llm_prompt_template = """You are an assistant for question-answering tasks.
Use the following context to answer the question.
If you don't know the answer, just say that you don't know.
Use five sentences maximum and keep the answer concise.\n
Question: {question} \nContext: {context} \nAnswer:"""

llm_prompt = PromptTemplate.from_template(llm_prompt_template)

print(llm_prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} template="You are an assistant for question-answering tasks.\nUse the following context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse five sentences maximum and keep the answer concise.\n\nQuestion: {question} \nContext: {context} \nAnswer:"


### Create a stuff documents chain

LangChain provides [Chains](https://python.langchain.com/docs/modules/chains/) for chaining together LLMs with each other or other components for complex applications. You will create a **stuff documents chain** for this application. A stuff documents chain lets you combine all the relevant documents, insert them into the prompt, and pass that prompt to the LLM.

You can create a stuff documents chain using the [LangChain Expression Language (LCEL)](https://python.langchain.com/docs/expression_language).

To learn more about different types of document chains, read LangChain's [chains guide](https://python.langchain.com/docs/modules/chains/document/).

The stuff documents chain for this application retrieves the relevant website data and passes it as the context to an LLM prompt along with the input question.

In [ ]:
# Combine data from documents to readable string format.
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Create stuff documents chain using LCEL.
#
# This is called a chain because you are chaining together different elements
# with the LLM. In the following example, to create the stuff chain, you will
# combine the relevant context from the website data matching the question, the
# LLM model, and the output parser together like a chain using LCEL.
#
# The chain implements the following pipeline:
# 1. Extract the website data relevant to the question from the Chroma
#    vector store and save it to the variable `context`.
# 2. `RunnablePassthrough` option to provide `question` when invoking
#    the chain.
# 3. The `context` and `question` are then passed to the prompt where they
#    are populated in the respective variables.
# 4. This prompt is then passed to the LLM (`gemini-2.0-flash`).
# 5. Output from the LLM is passed through an output parser
#    to structure the model's response.
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | llm_prompt
    | llm
    | StrOutputParser()
)

### Prompt the model

You can now query the LLM by passing any question to the `invoke()` function of the stuff documents chain you created previously.

In [ ]:
from IPython.display import Markdown

Markdown(rag_chain.invoke("What is Gemini?"))

Gemini is a highly flexible, natively multimodal AI model capable of running efficiently on everything from data centers to mobile devices. It is optimized in three different sizes: Gemini Ultra for highly complex tasks, Gemini Pro for scaling across a wide range of tasks, and Gemini Nano for on-device tasks. From the start, Gemini was trained to seamlessly understand, reason about, and combine different modalities, including text, images, audio, video, and mathematical concepts. Additionally, it can understand, explain, and generate high-quality code in popular programming languages like Python, Java, C++, and Go. Its state-of-the-art capabilities enable it to solve complex problems and perform sophisticated reasoning across various domains.

# Conclusion

That's it. You have successfully created an LLM application that answers questions using data from a website with the help of Gemini, LangChain, and Chroma.